# Research 64 — ChimeraBoost default OOF

**Hypothesis:** ChimeraBoost, a CatBoost-style booster written in pure Python, trains from scratch and may add diversity to the equal CatBoost/LightGBM reference. Run one fixed default configuration (no tuning) on the reference's three stratified folds (seed 42) with the tuned feature set. Predeclared decision: promote only if the fixed 80/20 blend beats the reference on macro F1 and is not worse on weighted F1. Train OOF only; Test is not loaded.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from chimeraboost import ChimeraBoostClassifier
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
train = pd.read_csv(ROOT / "data/train.csv")
y = train.segment.to_numpy(dtype=int)
ref = pd.read_csv(ROOT / "predictions/segment-oof.csv")
assert len(ref) == len(y) and np.array_equal(ref.segment.to_numpy(), y)
p_ref = ref[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
p_ref = p_ref / p_ref.sum(axis=1, keepdims=True)
folds = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(train, y))

In [2]:
def features(frame):
    x = frame.drop(
        columns=["segment", "player_id", "random_metric_1", "random_metric_2", "random_metric_3"],
        errors="ignore",
    ).copy()
    ratios = {
        "spend_per_hour": ("total_spending_thb", "total_playtime_hours"),
        "spend_per_account_day": ("total_spending_thb", "account_age_days"),
        "playtime_per_account_day": ("total_playtime_hours", "account_age_days"),
        "monthly_spend_per_purchase": ("avg_monthly_spending", "spending_frequency"),
        "gifts_per_friend": ("gifts_sent_received", "friend_count"),
        "invites_per_friend": ("friend_invites_sent", "friend_count"),
        "ranked_per_team": ("ranked_participation_rate", "team_play_percentage"),
    }
    for name, (a, b) in ratios.items():
        x[name] = x[a] / (x[b].abs() + 1)
    x["weekly_minutes"] = x["play_frequency"] * x["avg_session_duration"]
    x["social_activity"] = x["friend_count"] * x["team_play_percentage"]
    x["missing_count"] = x.isna().sum(axis=1)
    return x.replace([np.inf, -np.inf], np.nan)


X = features(train)
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]
print("features", X.shape[1], "categorical", len(cat_cols), "numeric", len(num_cols))

features 51 categorical 10 numeric 41


In [3]:
X_cb = X.copy()
X_cb[cat_cols] = X_cb[cat_cols].fillna("__MISSING__").astype(str)
p_cand = np.zeros((len(y), 4), dtype=float)
started = time.perf_counter()
for fold, (fit_idx, valid_idx) in enumerate(folds, start=1):
    model = ChimeraBoostClassifier(random_state=42)
    model.fit(X_cb.iloc[fit_idx], y[fit_idx], cat_features=cat_cols)
    p_cand[valid_idx] = model.predict_proba(X_cb.iloc[valid_idx])
    fold_f1 = f1_score(y[valid_idx], p_cand[valid_idx].argmax(1), average="macro")
    print(f"Fold {fold}: ChimeraBoost macro F1={fold_f1:.6f}", flush=True)
print(f"Elapsed seconds={time.perf_counter() - started:.1f}. Train OOF only; Test not loaded.")

Fold 1: ChimeraBoost macro F1=0.741670


Fold 2: ChimeraBoost macro F1=0.744642


Fold 3: ChimeraBoost macro F1=0.748199


Elapsed seconds=104.4. Train OOF only; Test not loaded.


In [4]:
def score_table(name, p_cand):
    ref_folds = [f1_score(y[v], p_ref[v].argmax(1), average="macro") for _, v in folds]
    rows = []
    for label, probs in [
        ("Reference equal CatBoost/LightGBM", p_ref),
        (f"{name} alone", p_cand),
        (f"Fixed 80/20 reference + {name}", 0.8 * p_ref + 0.2 * p_cand),
    ]:
        pred = probs.argmax(axis=1)
        row = {
            "candidate": label,
            "macro_f1": f1_score(y, pred, average="macro"),
            "weighted_f1": f1_score(y, pred, average="weighted"),
        }
        for k, (_, v) in enumerate(folds, start=1):
            row[f"fold_{k}_delta_vs_reference"] = (
                f1_score(y[v], pred[v], average="macro") - ref_folds[k - 1]
            )
        rows.append(row)
    return pd.DataFrame(rows)

In [5]:
results = score_table("ChimeraBoost", p_cand)
display(results.round(6))
results.to_csv(ROOT / "assets/segment-chimeraboost-results.csv", index=False)
pd.DataFrame(p_cand, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
    ROOT / "predictions/segment-chimeraboost-oof.csv", index=False
)

,candidate,macro_f1,weighted_f1,fold_1_delta_vs_reference,fold_2_delta_vs_reference,fold_3_delta_vs_reference
0,Reference equal CatBoost/LightGBM,0.750895,0.764683,0.000000,0.000000,0.000000
1,ChimeraBoost alone,0.744862,0.758868,-0.007775,-0.005179,-0.005190
2,Fixed 80/20 reference + ChimeraBoost,0.751660,0.765359,-0.000139,0.001244,0.001178


## Research decision

Decide on the fixed 80/20 row above using the predeclared rule. This is one configuration; no hyperparameter search was run.